# Sesión 1 — Stacking y Blending sin fuga

**Caso:** comité de modelos InmoValor

**Pregunta problema:** ¿podemos combinar modelos diferentes para reducir el error sin fabricar una mejora aparente?

Este notebook materializa promedio, blending y stacking. El test se separa al comienzo y no participa en la selección.

## Objetivos

1. Medir calidad y diversidad de modelos base.
2. Construir predicciones *out-of-fold* (OOF).
3. Comparar promedio y stacking con los mismos folds.
4. Mostrar el mecanismo de blending y su costo de datos.
5. Abrir test una vez después de congelar la selección.

In [1]:
from pathlib import Path
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, StackingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
CV = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")
warnings.filterwarnings("ignore", category=FutureWarning)

def locate_dataset():
    candidates = [
        Path("datasets/public/Ames_Housing.csv"),
        Path("../../datasets/public/Ames_Housing.csv"),
        Path("../datasets/public/Ames_Housing.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("No se encontró datasets/public/Ames_Housing.csv")

data_path = locate_dataset()
frame = pd.read_csv(data_path)
X = frame.drop(columns=["SalePrice", "Id"])
y = frame["SalePrice"].astype(float)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

def make_preprocessor(data):
    numeric = data.select_dtypes(include=np.number).columns.tolist()
    categorical = data.select_dtypes(exclude=np.number).columns.tolist()
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        # "Missing" conserva la semántica de amenidades ausentes y evita
        # eliminar columnas que estén completamente vacías dentro de un fold.
        ("imputer", SimpleImputer(
            strategy="constant", fill_value="Missing", keep_empty_features=True
        )),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipe, numeric),
        ("categorical", categorical_pipe, categorical),
    ])

def wrap(model, data=X_dev):
    return Pipeline([
        ("preprocess", make_preprocessor(data)),
        ("model", model),
    ])

print(f"Datos: {frame.shape[0]:,} viviendas, {X.shape[1]} predictores")
print(f"Desarrollo: {len(X_dev):,} | Test reservado: {len(X_test):,}")

Datos: 1,460 viviendas, 79 predictores
Desarrollo: 1,168 | Test reservado: 292


## 1. Contrato experimental

- Métrica principal: MAE en dólares.
- CV: tres folds barajados con semilla fija.
- Preprocesamiento: dentro de cada pipeline.
- Selección: solo con desarrollo.
- Test: auditoría final, una vez congelada la decisión.

In [2]:
baseline = DummyRegressor(strategy="median")
base_models = {
    "Ridge": wrap(Ridge(alpha=10.0)),
    "Random Forest": wrap(RandomForestRegressor(
        n_estimators=250, min_samples_leaf=2,
        random_state=RANDOM_STATE, n_jobs=1,
    )),
    "Gradient Boosting": wrap(GradientBoostingRegressor(
        n_estimators=180, learning_rate=0.05, max_depth=2,
        min_samples_leaf=8, random_state=RANDOM_STATE,
    )),
}

rows = []
baseline_scores = -cross_val_score(
    baseline, X_dev, y_dev, scoring="neg_mean_absolute_error", cv=CV
)
rows.append({"modelo": "Baseline mediana", "MAE_medio": baseline_scores.mean(),
             "MAE_sd": baseline_scores.std(ddof=1)})
for name, estimator in base_models.items():
    scores = -cross_val_score(
        estimator, X_dev, y_dev, scoring="neg_mean_absolute_error", cv=CV, n_jobs=1
    )
    rows.append({"modelo": name, "MAE_medio": scores.mean(),
                 "MAE_sd": scores.std(ddof=1)})

base_results = pd.DataFrame(rows).sort_values("MAE_medio")
base_results

,modelo,MAE_medio,MAE_sd
3,Gradient Boosting,"17,782.534","1,619.361"
1,Ridge,"18,496.785",783.316
2,Random Forest,"18,911.339",599.923
0,Baseline mediana,"54,547.639","2,711.807"


### Lectura

El MAE individual es solo la mitad de la historia. Un modelo algo menos preciso puede aportar si sus residuos contienen información diferente.

## 2. Predicciones OOF y diversidad

Cada fila será predicha por un modelo que no la usó para entrenar. Esas predicciones sí pueden alimentar una comparación honesta y un meta-modelo.

In [3]:
oof_predictions = pd.DataFrame(index=X_dev.index)
for name, estimator in base_models.items():
    oof_predictions[name] = cross_val_predict(
        estimator, X_dev, y_dev, cv=CV, n_jobs=1
    )

residuals = oof_predictions.apply(lambda column: y_dev - column)
residual_correlation = residuals.corr()
residual_correlation

,Ridge,Random Forest,Gradient Boosting
Ridge,1.000,0.731,0.802
Random Forest,0.731,1.000,0.906
Gradient Boosting,0.802,0.906,1.000


In [4]:
average_oof = oof_predictions.mean(axis=1)
average_mae = mean_absolute_error(y_dev, average_oof)
pd.DataFrame({
    "MAE_OOF": [mean_absolute_error(y_dev, oof_predictions[col])
                for col in oof_predictions.columns] + [average_mae]
}, index=list(oof_predictions.columns) + ["Promedio simple"]).sort_values("MAE_OOF")

,MAE_OOF
Promedio simple,"16,147.752"
Gradient Boosting,"17,782.498"
Ridge,"18,496.097"
Random Forest,"18,910.814"


## 3. Stacking con dos niveles de validación

`StackingRegressor` crea internamente predicciones OOF para ajustar el meta-modelo. Luego una CV externa estima el ensamble completo. Usamos RidgeCV como meta-modelo para controlar pesos extremos.

In [5]:
stack = StackingRegressor(
    estimators=[
        ("ridge", clone(base_models["Ridge"])),
        ("rf", clone(base_models["Random Forest"])),
        ("gb", clone(base_models["Gradient Boosting"])),
    ],
    final_estimator=RidgeCV(alphas=np.logspace(-2, 3, 12)),
    cv=3,
    passthrough=False,
    n_jobs=1,
)
started = time.perf_counter()
stack_scores = -cross_val_score(
    stack, X_dev, y_dev, scoring="neg_mean_absolute_error", cv=CV, n_jobs=1
)
stack_seconds = time.perf_counter() - started
stack_summary = pd.DataFrame({
    "modelo": ["Stacking"],
    "MAE_medio": [stack_scores.mean()],
    "MAE_sd": [stack_scores.std(ddof=1)],
    "segundos": [stack_seconds],
})
stack_summary

,modelo,MAE_medio,MAE_sd,segundos
0,Stacking,"16,578.048","1,278.737",33.629


In [6]:
comparison = pd.concat([
    base_results,
    pd.DataFrame({"modelo": ["Promedio simple"], "MAE_medio": [average_mae],
                  "MAE_sd": [np.nan]}),
    stack_summary[["modelo", "MAE_medio", "MAE_sd"]],
], ignore_index=True).sort_values("MAE_medio")
comparison.assign(
    mejora_vs_baseline_pct=lambda d: 100 * (
        float(base_results.loc[base_results.modelo.eq("Baseline mediana"), "MAE_medio"].iloc[0])
        - d.MAE_medio
    ) / float(base_results.loc[base_results.modelo.eq("Baseline mediana"), "MAE_medio"].iloc[0])
)

,modelo,MAE_medio,MAE_sd,mejora_vs_baseline_pct
4,Promedio simple,"16,147.752",NaN,70.397
5,Stacking,"16,578.048","1,278.737",69.608
0,Gradient Boosting,"17,782.534","1,619.361",67.400
1,Ridge,"18,496.785",783.316,66.091
2,Random Forest,"18,911.339",599.923,65.331
3,Baseline mediana,"54,547.639","2,711.807",0.000


## 4. Blending: implementación transparente

Se reserva 20 % de desarrollo para entrenar el meta-modelo. Esto evita predicciones *in-sample*, pero reduce los datos disponibles para los modelos base y depende de una sola partición.

In [7]:
X_base, X_meta, y_base, y_meta = train_test_split(
    X_dev, y_dev, test_size=0.20, random_state=RANDOM_STATE
)
blend_meta = pd.DataFrame(index=X_meta.index)
blend_test = pd.DataFrame(index=X_test.index)
fitted_base = {}
for name, estimator in base_models.items():
    fitted = clone(estimator).fit(X_base, y_base)
    fitted_base[name] = fitted
    blend_meta[name] = fitted.predict(X_meta)
    blend_test[name] = fitted.predict(X_test)

blender = RidgeCV(alphas=np.logspace(-2, 4, 20)).fit(blend_meta, y_meta)
blend_weights = pd.Series(blender.coef_, index=blend_meta.columns, name="peso")
pd.concat([blend_weights, pd.Series({"intercepto": blender.intercept_})])

Ridge                     0.401
Random Forest             0.567
Gradient Boosting         0.125
intercepto          -16,290.007
dtype: float64

La pérdida sobre `X_meta` no es una estimación final: esas respuestas entrenaron el blender. El ejemplo muestra el mecanismo; una evaluación repetida requeriría repetir todo el procedimiento dentro de folds externos.

## 5. Congelar la selección y abrir test

Elegimos entre modelos evaluados por CV. Si stacking no supera al mejor individual, conservamos la solución más simple.

In [8]:
cv_candidates = comparison.dropna(subset=["MAE_sd"]).copy()
selected_name = cv_candidates.iloc[0]["modelo"]
if selected_name == "Stacking":
    selected = stack
else:
    selected = base_models[selected_name]

selected.fit(X_dev, y_dev)
test_prediction = selected.predict(X_test)
selected_test_mae = mean_absolute_error(y_test, test_prediction)

# Blending se informa como contraste, no como candidato elegido por su error in-sample.
blend_test_mae = mean_absolute_error(y_test, blender.predict(blend_test))
pd.DataFrame({
    "procedimiento": [selected_name, "Blending ilustrativo"],
    "MAE_test": [selected_test_mae, blend_test_mae],
})

,procedimiento,MAE_test
0,Stacking,"16,867.725"
1,Blending ilustrativo,"16,771.389"


## Interpretación obligatoria

Complete antes de continuar:

- **Observo:** ¿stacking mejora frente al mejor individual y cuánto?
- **Significa:** ¿la mejora supera la variación entre folds?
- **Recomendaría:** ¿qué procedimiento llevaría a producción?
- **Vigilaría:** ¿qué costo o segmento revisaría?
- **No puedo concluir:** el mejor resultado observado no garantiza superioridad futura ni causalidad.

## Reto guiado

1. Active `passthrough=True` y anticipe el efecto antes de ejecutar.
2. Retire el par de modelos con residuos más correlacionados.
3. Compare cambio en MAE, variación y tiempo.
4. Defienda si la complejidad adicional se justifica.